# Low-Resource Neural Machine Translation: English → Malayalam

**Task:** Fine-tune a multilingual MarianMT model (`Helsinki-NLP/opus-mt-en-dra`) on English–Malayalam parallel data from the AI4Bharat BPCC corpus, then adapt it to scientific/ML domain text using synthetic arXiv abstract translations.

**Model:** `opus-mt-en-dra` — a Marian transformer trained on Dravidian languages (Malayalam, Kannada, Tamil, Telugu). A multilingual tokeniser covers all four scripts with a shared vocabulary of ~63k tokens.

**Data:** AI4Bharat BPCC `bpcc-seed-latest`, Malayalam split (~human-annotated parallel sentences). Synthetic domain data generated from ML arXiv abstracts.

**Results (Part 1 — Baseline):**

| Epoch | Train Loss | Val Loss | BLEU |
|-------|-----------|----------|------|
| 1 | 1.683 | 1.467 | 16.38 |
| 2 | 1.354 | 1.339 | 18.80 |
| 3 | 1.187 | 1.302 | 19.58 |
| 4 | 0.872 | 1.329 | 19.68 |
| 5 | 0.924 | 1.312 | 20.25 |

Validation BLEU (clean inference): **15.48**

Part 2 (domain adaptation on arXiv synthetic data) is in progress.

## 1. Setup

Mount Google Drive first — all checkpoints and saved models go there to survive runtime disconnections.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q sacrebleu datasets transformers sentencepiece

In [ ]:
import os
import re
import json
import random
import numpy as np
import torch

# Reproducibility
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 2. Data: AI4Bharat BPCC

The [Bharat Parallel Corpus Collection (BPCC)](https://huggingface.co/datasets/ai4bharat/BPCC) is a large-scale parallel corpus covering all 22 scheduled Indian languages. We use the `bpcc-seed-latest` config — the highest-quality subset, comprising human-annotated English–Indic sentence pairs.

The dataset is indexed by language code rather than a conventional train/test split. We extract the Malayalam split (`mal_Mlym`) and perform our own 80/20 train–validation split.

In [ ]:
from datasets import load_dataset

bpcc = load_dataset(
    "ai4bharat/BPCC",
    "bpcc-seed-latest",
    trust_remote_code=True,
)

full_data = bpcc["mal_Mlym"]
print(f"Total examples: {len(full_data)}")
print(full_data[0])

In [ ]:
split      = full_data.train_test_split(test_size=0.2, seed=SEED)
train_data = split["train"]
val_data   = split["test"]

print(f"Train: {len(train_data):,}  |  Validation: {len(val_data):,}")

## 3. Model: Helsinki-NLP/opus-mt-en-dra

`opus-mt-en-dra` is a MarianMT encoder-decoder transformer pre-trained on English → Dravidian language pairs. We use it rather than a Malayalam-specific model because:

- Its multilingual tokeniser handles Malayalam script natively
- Shared Dravidian vocabulary enables transfer across related languages
- It is compact (~300M parameters) and fine-tunable on a single T4 GPU

**Multilingual tokenisation:** Marian uses a SentencePiece tokeniser with a unified vocabulary across all supported scripts. Language selection is via a special prefix tag prepended to the source sentence (e.g. `>>mal<<` for Malayalam). This tells the decoder which language to generate in.

**Missing keys warning:** When loading, PyTorch warns about `embed_positions.weight` and `lm_head.weight`. These are tied weights reconstructed from the embedding matrix at runtime — not actually missing. Safe to ignore.

In [ ]:
from transformers import MarianMTModel, MarianTokenizer

MODEL_ID = "Helsinki-NLP/opus-mt-en-dra"
tokeniser = MarianTokenizer.from_pretrained(MODEL_ID)
model     = MarianMTModel.from_pretrained(MODEL_ID)
model     = model.to(device)

print(f"Vocab size:  {tokeniser.vocab_size:,}")
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

## 4. Preprocessing and Tokenisation

**Tokenisation for seq2seq:** Unlike classification, we tokenise both source and target sequences. The source gets the `>>mal<<` language tag prepended. The target is tokenised separately and stored as `labels`. During training, the model learns to predict the next target token given the source and all previous target tokens (teacher forcing).

**Truncation at 256 tokens:** Malayalam sentences are morphologically complex — a single word can encode what English expresses in several words. We cap at 256 tokens, which covers ~98% of sentences in this corpus without truncation.

**No padding here:** Padding is deferred to the data collator, which pads dynamically per batch to the length of the longest sequence in that batch. This is more efficient than padding to a global maximum.

In [ ]:
MAX_LEN = 256

def preprocess_batch(batch):
    src = [f">>mal<< {s}" for s in batch["src"]]
    tgt = batch["tgt"]

    model_inputs = tokeniser(
        src,
        max_length=MAX_LEN,
        truncation=True,
        padding=False,
    )

    labels = tokeniser(
        text_target=tgt,
        max_length=MAX_LEN,
        truncation=True,
        padding=False,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenised = {}
for name, data in [("train", train_data), ("validation", val_data)]:
    tokenised[name] = data.map(
        preprocess_batch,
        batched=True,
        remove_columns=data.column_names,
    )
    print(f"{name}: {len(tokenised[name]):,} examples tokenised")

## 5. Fine-tuning

**Data collator:** Pads each batch dynamically to the longest sequence in that batch. `pad_to_multiple_of=8` aligns tensor dimensions to multiples of 8, which improves throughput on Tensor Core GPUs (T4, A100). The collator also replaces padding token IDs in `labels` with `-100`, which tells PyTorch's cross-entropy loss to ignore those positions — we only penalise the model for tokens it was actually supposed to predict.

**BLEU as training metric:** We use corpus BLEU (via `sacrebleu`) computed on beam-search decoded outputs at the end of each epoch. BLEU measures n-gram overlap between hypothesis and reference translations. It is fast to compute and correlates well with human judgement at the corpus level, making it the standard MT training signal.

**Batching:** Training processes 32 sentence pairs at a time. Each batch is a forward pass (encoder encodes source, decoder generates target auto-regressively), followed by a backward pass (gradients flow through the decoder, cross-attention layers, and encoder). Larger batches are more GPU-efficient but require more memory.

**Autoregressive decoding:** During evaluation, the decoder generates one token at a time, conditioning each new token on all previously generated tokens. We use beam search with `num_beams=4`, maintaining the 4 most probable partial sequences at each step and returning the globally highest-probability complete sequence.

In [ ]:
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
import sacrebleu
import numpy as np

data_collator = DataCollatorForSeq2Seq(
    tokeniser,
    model=model,
    padding=True,
    pad_to_multiple_of=8,
)

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    labels = np.where(labels != -100, labels, tokeniser.pad_token_id)
    decoded_preds  = tokeniser.batch_decode(preds,  skip_special_tokens=True)
    decoded_labels = tokeniser.batch_decode(labels, skip_special_tokens=True)
    decoded_labels = [[l] for l in decoded_labels]
    result = sacrebleu.corpus_bleu(decoded_preds, decoded_labels)
    return {"bleu": round(result.score, 2)}

training_args = Seq2SeqTrainingArguments(
    output_dir="/content/drive/MyDrive/marian",   # save directly to Drive
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=500,
    weight_decay=0.01,
    learning_rate=5e-5,
    fp16=True,
    predict_with_generate=True,
    generation_max_length=256,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    logging_steps=100,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenised["train"],
    eval_dataset=tokenised["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch	Training Loss	Validation Loss	BLEU
1	1.683142	1.467089	16.38
2	1.353829	1.338669	18.80
3	1.187118	1.301803	19.58
4	0.872330	1.328729	19.68
5	0.924329	1.312344	20.25


## 6. Save Baseline Model

In [ ]:
model.save_pretrained("/content/drive/MyDrive/marian_bpcc_final")
tokeniser.save_pretrained("/content/drive/MyDrive/marian_bpcc_final")
print(os.listdir("/content/drive/MyDrive/marian_bpcc_final"))

## 7. Inference and Baseline Evaluation

**Inference bottlenecks:** Autoregressive generation is inherently sequential — each token depends on the previous one, so the decoder cannot be parallelised across the sequence dimension. Throughput is primarily limited by:
1. Memory bandwidth (loading model weights per step)
2. Sequence length (longer outputs = more sequential steps)
3. Beam width (4 beams = 4× the computation of greedy decoding)

Batching the source sentences (processing 32 at a time) amortises the encoder cost, since the encoder runs in parallel across the batch.

In [ ]:
from transformers import MarianMTModel, MarianTokenizer

model     = MarianMTModel.from_pretrained("/content/drive/MyDrive/marian_bpcc_final")
tokeniser = MarianTokenizer.from_pretrained("/content/drive/MyDrive/marian_bpcc_final")
model     = model.to(device)

def translate_batch(texts, target_lang=">>mal<<", batch_size=32):
    results = []
    for i in range(0, len(texts), batch_size):
        batch  = [f"{target_lang} {t}" for t in texts[i:i+batch_size]]
        inputs = tokeniser(batch, return_tensors="pt", padding=True,
                           truncation=True, max_length=256)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            generated = model.generate(**inputs, num_beams=4, max_length=256)
        decoded = tokeniser.batch_decode(generated, skip_special_tokens=True)
        results.extend(decoded)
    return results

# Sanity check
print(translate_batch(["Where is the nearest hospital?"])[0])
print(translate_batch(["We propose a novel attention mechanism for neural machine translation."])[0])

**BLEU limitations:** BLEU measures n-gram overlap between hypothesis and reference. It has well-known failure modes:
- It rewards surface-level lexical matches, not semantic equivalence
- It penalises valid paraphrases that don't overlap with the reference
- Low-resource languages like Malayalam have rich morphology — a correctly inflected form of a word may score 0 if the reference uses a different but equally valid inflection
- It does not account for word order differences between typologically different languages (English is SVO; Malayalam is SOV)

For this project, BLEU is used as a relative metric (baseline vs. adapted model) rather than an absolute quality score.

In [ ]:
from sacrebleu.metrics import BLEU

bleu = BLEU()

sources    = val_data['src']
references = val_data['tgt']

hypotheses = translate_batch(sources)
score      = bleu.corpus_score(hypotheses, [references])
print(f"Baseline BLEU: {score}")
# Result: BLEU = 15.48 47.6/20.8/10.4/5.6 (BP=0.999)

Baseline BLEU: BLEU = 15.48 47.6/20.8/10.4/5.6 (BP = 0.999 ratio = 0.999 hyp_len = 243341 ref_len = 243647)


## 8. Synthetic arXiv Domain Data (Part 2 — In Progress)

**Domain adaptation:** The BPCC corpus is general-domain (news, Wikipedia, everyday text). ML/scientific text introduces vocabulary that is either absent from or rare in the training data — terms like *gradient descent*, *attention mechanism*, *latent space*, *fine-tuning*. The model handles these poorly, either transliterating them phonetically or mapping them to semantically adjacent but incorrect Malayalam words.

To adapt the model to this domain, we generate synthetic parallel data: take ML arXiv abstracts (English), translate them with the baseline model, and use the resulting English–Malayalam pairs as additional training data. The model then fine-tunes on a mixture of BPCC (general domain) and arXiv synthetic (scientific domain).

**Known limitation of v1 synthetic data:** The abstracts contain LaTeX math notation (`$\epsilon$`, `\mathcal{L}`) which the model has never seen. These get transliterated as garbage. v2 will strip LaTeX before translation.

In [ ]:
import re

def clean_abstract(text):
    """Remove LaTeX notation from arXiv abstracts before translation."""
    text = re.sub(r'\$.*?\$', '', text)              # inline math: $...$
    text = re.sub(r'\$\$.*?\$\$', '', text)          # display math: $$...$$
    text = re.sub(r'\\[a-zA-Z]+\{.*?\}', '', text)  # commands: \cmd{...}
    text = re.sub(r'\\[a-zA-Z]+', '', text)           # bare commands: \cmd
    text = re.sub(r'\s+', ' ', text).strip()
    return text

from datasets import load_dataset

arxiv  = load_dataset("CShorten/ML-ArXiv-Papers", split="train")
sample = arxiv.shuffle(seed=SEED).select(range(500))

# Combine title + abstract, clean LaTeX
clean_texts = []
for row in sample:
    title    = clean_abstract(row['title'])
    abstract = clean_abstract(row['abstract'])
    combined = f"{title}. {abstract}".strip()
    if len(combined) > 50:   # drop entries that are mostly math
        clean_texts.append(combined)

print(f"Clean abstracts: {len(clean_texts)}")
print(clean_texts[0][:300])

In [ ]:
synthetic_ml = translate_batch(clean_texts)

synthetic = [{"src": en, "tgt": ml}
             for en, ml in zip(clean_texts, synthetic_ml)]

with open("/content/drive/MyDrive/arxiv_synthetic_en_ml_v2.json", "w", ensure_ascii=False) as f:
    json.dump(synthetic, f)

print(f"Saved {len(synthetic)} pairs")

*Domain adaptation fine-tuning (combining BPCC train data with 3× upsampled arXiv synthetic pairs) to follow in Part 2.*